<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>5. Code Generation and Explanation</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import anthropic
from IPython.display import display, Markdown, Code

import matplotlib.pyplot as plt

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 87485bac20625f43da5809d38ec119aca4321f71

IPython   : 9.8.0
matplotlib: 3.10.7
watermark : 2.5.0
anthropic : 0.75.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

Initialize the client and define utility functions

In [4]:
client = anthropic.Anthropic()

def show_response(response):
    """Display Claude's response with markdown formatting."""
    display(Markdown(response.content[0].text))

def show_json(content):
    print(json.dumps(content, indent=2))

def show_code(response, language="python"):
    """Display code with syntax highlighting."""
    display(Markdown(f"```{language}\n{response.content[0].text}\n```"))

## Generating Code from Prompts

Claude excels at generating code from natural language descriptions. The key is providing clear, specific requirements.

In [5]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are an expert Python developer. Write clean, well-documented code.",
    messages=[
        {
            "role": "user",
            "content": "Write a Python function that checks if a string is a valid email address using regex."
        }
    ]
)

In [6]:
show_response(response)

Here's a Python function that validates email addresses using regex:

```python
import re

def is_valid_email(email):
    """
    Check if a string is a valid email address using regex.
    
    This function validates email addresses based on RFC 5322 standards
    with some practical simplifications. It checks for:
    - Local part: alphanumeric characters, dots, hyphens, underscores, plus signs
    - @ symbol separating local and domain parts
    - Domain part: alphanumeric characters, dots, hyphens
    - Top-level domain: at least 2 characters
    
    Args:
        email (str): The email string to validate
        
    Returns:
        bool: True if the email is valid, False otherwise
        
    Examples:
        >>> is_valid_email("user@example.com")
        True
        >>> is_valid_email("invalid.email")
        False
        >>> is_valid_email("user@sub.example.org")
        True
    """
    if not isinstance(email, str):
        return False
    
    # Email regex pattern
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    
    # Additional checks for edge cases
    if email.count('@') != 1:
        return False
    
    # Check if email starts or ends with a dot
    if email.startswith('.') or email.endswith('.'):
        return False
    
    # Check for consecutive dots
    if '..' in email:
        return False
    
    # Check if local part starts or ends with a dot
    local_part = email.split('@')[0]
    if local_part.startswith('.') or local_part.endswith('.'):
        return False
    
    return bool(re.match(pattern, email))


def is_valid_email_strict(email):
    """
    A more comprehensive email validation function with stricter rules.
    
    This version includes additional validation for:
    - Length limits (local part ≤ 64 chars, domain ≤ 253 chars)
    - Domain part must contain at least one dot
    - More restrictive character validation
    
    Args:
        email (str): The email string to validate
        
    Returns:
        bool: True if the email is valid, False otherwise
    """
    if not isinstance(email, str):
        return False
    
    # Length check (total email should be ≤ 320 characters)
    if len(email) > 320:
        return False
    
    # Must contain exactly one @ symbol
    if email.count('@') != 1:
        return False
    
    local_part, domain_part = email.split('@')
    
    # Length checks for parts
    if len(local_part) == 0 or len(local_part) > 64:
        return False
    if len(domain_part) == 0 or len(domain_part) > 253:
        return False
    
    # Local part validation
    local_pattern = r'^[a-zA-Z0-9._%+-]+$'
    if not re.match(local_pattern, local_part):
        return False
    
    # Local part cannot start or end with a dot
    if local_part.startswith('.') or local_part.endswith('.'):
        return False
    
    # No consecutive dots in local part
    if '..' in local_part:
        return False
    
    # Domain part validation
    domain_pattern = r'^[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    if not re.match(domain_pattern, domain_part):
        return False
    
    # Domain part cannot start or end with a dot or hyphen
    if (domain_part.startswith('.') or domain_part.endswith('.') or
        domain_part.startswith('-') or domain_part.endswith('-')):
        return False
    
    # No consecutive dots in domain part
    if '..' in domain_part:
        return False
    
    return True


# Example usage and test cases
if __name__ == "__main__":
    #

The more specific the requirements/prompt the better the outcome

In [7]:
detailed_prompt = """
Write a Python class called `RateLimiter` with the following specifications:

1. Constructor takes `max_requests` (int) and `time_window` (int, seconds)
2. Method `is_allowed()` returns True if request is within rate limit
3. Method `reset()` clears the request history
4. Use a sliding window algorithm
5. Include type hints
6. Add docstrings

Example usage:
```python
limiter = RateLimiter(max_requests=5, time_window=60)
if limiter.is_allowed():
    # Process request
```
"""

In [8]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are an expert Python developer. Write production-quality code.",
    messages=[{"role": "user", "content": detailed_prompt}]
)

show_response(response)

```python
from typing import List
from collections import deque
import time


class RateLimiter:
    """
    A rate limiter implementation using a sliding window algorithm.
    
    This class tracks requests within a specified time window and enforces
    a maximum number of requests allowed during that period.
    
    Attributes:
        max_requests (int): Maximum number of requests allowed in the time window
        time_window (int): Time window in seconds
    """
    
    def __init__(self, max_requests: int, time_window: int) -> None:
        """
        Initialize the RateLimiter.
        
        Args:
            max_requests (int): Maximum number of requests allowed in the time window.
                              Must be a positive integer.
            time_window (int): Time window in seconds. Must be a positive integer.
            
        Raises:
            ValueError: If max_requests or time_window are not positive integers.
        """
        if not isinstance(max_requests, int) or max_requests <= 0:
            raise ValueError("max_requests must be a positive integer")
        if not isinstance(time_window, int) or time_window <= 0:
            raise ValueError("time_window must be a positive integer")
            
        self.max_requests: int = max_requests
        self.time_window: int = time_window
        self._request_times: deque[float] = deque()
    
    def is_allowed(self) -> bool:
        """
        Check if a new request is allowed based on the rate limit.
        
        This method uses a sliding window algorithm to determine if the current
        request should be allowed. It automatically removes expired requests
        from the tracking window and adds the current request if it's within
        the rate limit.
        
        Returns:
            bool: True if the request is allowed (within rate limit),
                  False if the request should be rejected.
        """
        current_time = time.time()
        
        # Remove requests that are outside the current time window
        self._cleanup_expired_requests(current_time)
        
        # Check if we can accept a new request
        if len(self._request_times) < self.max_requests:
            self._request_times.append(current_time)
            return True
        
        return False
    
    def reset(self) -> None:
        """
        Clear the request history.
        
        This method removes all tracked requests, effectively resetting
        the rate limiter to its initial state.
        """
        self._request_times.clear()
    
    def _cleanup_expired_requests(self, current_time: float) -> None:
        """
        Remove requests that are outside the current time window.
        
        Args:
            current_time (float): The current timestamp to compare against.
        """
        cutoff_time = current_time - self.time_window
        
        # Remove expired requests from the left of the deque
        while self._request_times and self._request_times[0] <= cutoff_time:
            self._request_times.popleft()
    
    def get_remaining_requests(self) -> int:
        """
        Get the number of requests remaining in the current window.
        
        Returns:
            int: Number of additional requests that can be made within
                 the current time window.
        """
        current_time = time.time()
        self._cleanup_expired_requests(current_time)
        return max(0, self.max_requests - len(self._request_times))
    
    def get_reset_time(self) -> float:
        """
        Get the timestamp when the rate limit will reset.
        
        Returns:
            float: Timestamp when the oldest request in the current window
                   will expire, or current time if no requests are tracked.
        """
        if not self._request_times:
            return time.time()
        return self._request_times[0] + self.time_window
    
    def __repr__(self) -> str:
        """
        Return a string representation of the RateLimiter.
        
        Returns:
            str: String representation showing the current configuration.
        """
        return (f"RateLimiter(max_requests={self.max_requests}, "
                f"time_window={self.time_window})")


# Example usage and demonstration
if __name__ == "__main__":
    # Create a rate limiter allowing 5 requests per 60 seconds
    limiter = RateLimiter(max_requests=5, time_window=60)
    
    # Simulate requests
    for i in range(7):
        if limiter.is_allowed():
            print(f"Request {i + 1}: Allowed")
        else:
            print(f"Request {i + 1}: Rejected (rate limit exceeded)")
    
    print(f"Remaining requests: {limiter.get_remaining_requests()}")
    print(f"Rate limit resets at: {limiter.get_reset_time()}")
    
    # Reset the limiter
    limiter.reset()
    print("Rate limiter reset")
    print(f"Remaining requests after reset: {limiter.get_remaining_requests()}")
```

This implementation provides:

1. **Sliding Window Algorithm**: Uses a `deque` to efficiently track request timestamps and automatically removes expired requests.

2. **Type Hints**: Complete type annotations for all methods and attributes.

3. **Comprehensive Docstrings**: Detailed documentation following Google/NumPy style conventions.

4. **Input Validation**: Validates constructor parameters to ensure they are positive integers.

5. **Additional Utility Methods**: 
   - `get_remaining_requests()`: Shows how many requests are still available
   - `get_reset_time()`: Shows when the rate limit will reset
   - `__repr__()`: Provides a useful string representation

6. **Thread Safety Considerations**: While this basic implementation isn't thread-safe, it's designed to be easily extended with locks if needed.

7. **Efficient Cleanup**: The `_cleanup_expired_requests()` method ensures that expired requests are automatically removed, maintaining optimal performance.

The sliding window algorithm ensures that the rate limit is enforced over any sliding time window, providing more accurate rate limiting compared to fixed-window approaches.

### Generating Code in Multiple Languages


Claude can generate code in many languages

In [9]:
languages = ["Python", "JavaScript", "TypeScript", "Rust", "Go"]

task = "Write a function to reverse a string"

print("Same task in multiple languages:\n")

for lang in languages:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"{task}. Use {lang}. Just the code, no explanation."
        }]
    )
    
    print(f"{lang}:")
    show_response(response)
    print("\n" + "-"*50 + "\n")

Same task in multiple languages:

Python:


```python
def reverse_string(s):
    return s[::-1]
```


--------------------------------------------------

JavaScript:


```javascript
function reverseString(str) {
    return str.split('').reverse().join('');
}
```


--------------------------------------------------

TypeScript:


...


--------------------------------------------------

Rust:


```rust
fn reverse_string(s: &str) -> String {
    s.chars().rev().collect()
}
```


--------------------------------------------------

Go:


```go
func reverse(s string) string {
    runes := []rune(s)
    for i, j := 0, len(runes)-1; i < j; i, j = i+1, j-1 {
        runes[i], runes[j] = runes[j], runes[i]
    }
    return string(runes)
}
```


--------------------------------------------------



### SQL queries from natural language

In [10]:
sql_prompt = """
I have a database with the following tables:
- users (id, name, email, created_at, subscription_tier)
- orders (id, user_id, total_amount, status, created_at)
- products (id, name, price, category)
- order_items (order_id, product_id, quantity)

Write SQL queries for:
1. Get top 10 customers by total spending
2. Find products that have never been ordered
3. Calculate monthly revenue for the past year
4. Find users who haven't ordered in the last 30 days
"""

In [11]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are a database expert. Write efficient, readable SQL queries.",
    messages=[{"role": "user", "content": sql_prompt}]
)

show_response(response)

Here are efficient SQL queries for each of your requirements:

## 1. Top 10 customers by total spending

```sql
SELECT 
    u.id,
    u.name,
    u.email,
    SUM(o.total_amount) as total_spent
FROM users u
INNER JOIN orders o ON u.id = o.user_id
WHERE o.status NOT IN ('cancelled', 'refunded') -- Exclude cancelled/refunded orders
GROUP BY u.id, u.name, u.email
ORDER BY total_spent DESC
LIMIT 10;
```

## 2. Products that have never been ordered

```sql
SELECT 
    p.id,
    p.name,
    p.price,
    p.category
FROM products p
LEFT JOIN order_items oi ON p.id = oi.product_id
WHERE oi.product_id IS NULL
ORDER BY p.name;
```

## 3. Monthly revenue for the past year

```sql
SELECT 
    DATE_FORMAT(o.created_at, '%Y-%m') as month,
    SUM(o.total_amount) as monthly_revenue,
    COUNT(o.id) as order_count
FROM orders o
WHERE o.created_at >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
    AND o.status NOT IN ('cancelled', 'refunded')
GROUP BY DATE_FORMAT(o.created_at, '%Y-%m')
ORDER BY month DESC;
```

**Alternative for PostgreSQL:**
```sql
SELECT 
    TO_CHAR(o.created_at, 'YYYY-MM') as month,
    SUM(o.total_amount) as monthly_revenue,
    COUNT(o.id) as order_count
FROM orders o
WHERE o.created_at >= CURRENT_DATE - INTERVAL '1 year'
    AND o.status NOT IN ('cancelled', 'refunded')
GROUP BY TO_CHAR(o.created_at, 'YYYY-MM')
ORDER BY month DESC;
```

## 4. Users who haven't ordered in the last 30 days

```sql
SELECT 
    u.id,
    u.name,
    u.email,
    u.subscription_tier,
    MAX(o.created_at) as last_order_date
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
GROUP BY u.id, u.name, u.email, u.subscription_tier
HAVING MAX(o.created_at) < DATE_SUB(CURDATE(), INTERVAL 30 DAY) 
    OR MAX(o.created_at) IS NULL -- Include users who never ordered
ORDER BY last_order_date DESC;
```

**Alternative approach (potentially more efficient for large datasets):**
```sql
SELECT 
    u.id,
    u.name,
    u.email,
    u.subscription_tier
FROM users u
WHERE u.id NOT IN (
    SELECT DISTINCT o.user_id 
    FROM orders o 
    WHERE o.created_at > DATE_SUB(CURDATE(), INTERVAL 30 DAY)
        AND o.user_id IS NOT NULL
);
```

## Performance Notes:

1. **Indexes recommended:**
   - `orders(user_id, created_at, status)`
   - `order_items(product_id)`
   - `orders(created_at)` for time-based queries

2. **For PostgreSQL users:** Replace `DATE_SUB(CURDATE(), INTERVAL X)` with `CURRENT_DATE - INTERVAL 'X'`

3. **For very large datasets:** Consider adding date range filters to limit the scope of calculations where appropriate.

## Explaining Existing Code

Some code to explain

In [12]:
complex_code = '''
def memoize(func):
    cache = {}
    def wrapper(*args, **kwargs):
        key = str(args) + str(sorted(kwargs.items()))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
'''

In [13]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Explain this Python code in detail. Cover:
1. What each part does
2. How the decorator pattern works
3. Why memoization helps here
4. The time complexity improvement

```python
{complex_code}
```"""
    }]
)

In [14]:
show_response(response)

I'll break down this memoization decorator and explain how it dramatically improves the Fibonacci function's performance.

## 1. What Each Part Does

### The Memoization Decorator
```python
def memoize(func):
    cache = {}  # Dictionary to store previously computed results
    def wrapper(*args, **kwargs):
        # Create a unique key from function arguments
        key = str(args) + str(sorted(kwargs.items()))
        
        # Check if we've already computed this result
        if key not in cache:
            cache[key] = func(*args, **kwargs)  # Compute and store
        
        return cache[key]  # Return cached result
    return wrapper
```

### The Fibonacci Function
```python
@memoize  # Apply the memoization decorator
def fibonacci(n):
    if n < 2:
        return n  # Base cases: fib(0)=0, fib(1)=1
    return fibonacci(n - 1) + fibonacci(n - 2)  # Recursive case
```

## 2. How the Decorator Pattern Works

The decorator pattern transforms the original function:

```python
# Before decoration (conceptually):
def fibonacci_original(n):
    if n < 2:
        return n
    return fibonacci_original(n - 1) + fibonacci_original(n - 2)

# After decoration:
fibonacci = memoize(fibonacci_original)
```

**Step-by-step process:**
1. `@memoize` is syntactic sugar for `fibonacci = memoize(fibonacci)`
2. `memoize()` returns the `wrapper` function
3. When `fibonacci(5)` is called, it actually calls `wrapper(5)`
4. The wrapper checks the cache before calling the original function

## 3. Why Memoization Helps Here

### The Problem with Naive Fibonacci
Without memoization, calculating `fibonacci(5)` creates this call tree:
```
fibonacci(5)
├── fibonacci(4)
│   ├── fibonacci(3)
│   │   ├── fibonacci(2)
│   │   │   ├── fibonacci(1) → 1
│   │   │   └── fibonacci(0) → 0
│   │   └── fibonacci(1) → 1
│   └── fibonacci(2)
│       ├── fibonacci(1) → 1  # DUPLICATE!
│       └── fibonacci(0) → 0  # DUPLICATE!
└── fibonacci(3)               # ENTIRE SUBTREE DUPLICATED!
    ├── fibonacci(2)
    │   ├── fibonacci(1) → 1
    │   └── fibonacci(0) → 0
    └── fibonacci(1) → 1
```

**Notice the massive duplication!** `fibonacci(2)` is calculated 3 times, `fibonacci(1)` is calculated 5 times.

### With Memoization
```python
# First call to fibonacci(5):
fibonacci(4) # Computed and cached
fibonacci(3) # Computed and cached  
fibonacci(2) # Computed and cached
fibonacci(1) # Computed and cached
fibonacci(0) # Computed and cached

# When fibonacci(3) is needed again:
# Returns cached value instantly!
```

## 4. Time Complexity Improvement

### Without Memoization: O(2^n)
- Each call spawns 2 recursive calls
- Forms a binary tree of height n
- Total calls ≈ 2^n

```python
# Exponential growth:
fibonacci(10): ~1,024 calls
fibonacci(20): ~1,048,576 calls  
fibonacci(30): ~1,073,741,824 calls (over 1 billion!)
```

### With Memoization: O(n)
- Each unique value from 0 to n is computed exactly once
- Subsequent calls return cached results in O(1)
- Total unique computations = n + 1

```python
# Linear growth:
fibonacci(10): 11 calls
fibonacci(20): 21 calls
fibonacci(30): 31 calls
```

## Practical Demonstration

```python
import time

def time_function(

Explaining code for different audiences

In [15]:
async_code = '''
async def fetch_all(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_one(session, url) for url in urls]
        return await asyncio.gather(*tasks)

async def fetch_one(session, url):
    async with session.get(url) as response:
        return await response.json()
'''

In [16]:
audiences = [
    ("beginner", "Explain to someone new to programming. Use simple analogies."),
    ("intermediate", "Explain to someone familiar with Python but new to async."),
    ("expert", "Focus on performance implications and best practices.")
]

print("Same code, different audiences:\n")

for level, instruction in audiences:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction}

```python
{async_code}
```"""
        }]
    )
    
    print(f"  {level.upper()} Explanation:")
    print(response.content[0].text[:500] + "..." if len(response.content[0].text) > 500 else response.content[0].text)
    print("\n" + "="*60 + "\n")

Same code, different audiences:

  BEGINNER Explanation:
I'll explain this code using simple analogies that relate to everyday life!

## The Restaurant Analogy

Think of this code like **ordering food from multiple restaurants at the same time** instead of waiting in line at each one.

### The Old Way (Synchronous)
Imagine you want food from 5 different restaurants. The slow way would be:
1. Drive to Restaurant A, order, wait for food ⏰
2. Drive to Restaurant B, order, wait for food ⏰  
3. Drive to Restaurant C, order, wait for food ⏰
4. ...and so on
...


  INTERMEDIATE Explanation:
Great question! Let me break down async programming and this code for you.

## What is Async Programming?

Think of async as a way to handle **waiting** more efficiently. In regular Python, when you make a web request, your program stops and waits for the response before continuing. With async, your program can start multiple requests and work on other things while waiting.

**Regular Python (synchronous):

Review code for vulnerabilities and other issues

In [17]:
code_to_review = '''
def get_user_data(user_id):
    connection = mysql.connector.connect(host="localhost", user="root", password="password123", database="mydb")
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM users WHERE id = " + str(user_id))
    result = cursor.fetchone()
    return result
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1200,
    messages=[{
        "role": "user",
        "content": f"""Review this code for:
1. Security vulnerabilities
2. Best practices violations
3. Error handling issues
4. Provide an improved version

```python
{code_to_review}
```"""
    }]
)

show_response(response)

## Code Review Issues

### 1. Security Vulnerabilities
- **SQL Injection**: Direct string concatenation allows malicious input to execute arbitrary SQL commands
- **Hardcoded credentials**: Database password is exposed in source code
- **No input validation**: `user_id` parameter accepts any input without verification

### 2. Best Practices Violations
- **No connection management**: Database connections aren't properly closed, causing resource leaks
- **Hardcoded configuration**: Database connection details should be externalized
- **No abstraction**: Direct database access without proper data access layer
- **Missing documentation**: No docstring or type hints

### 3. Error Handling Issues
- **No exception handling**: Database errors will crash the application
- **Silent failures**: Connection issues or query problems aren't handled
- **No graceful degradation**: No fallback mechanism for failures

## Improved Version

```python
import mysql.connector
from mysql.connector import Error
from contextlib import contextmanager
import os
from typing import Optional, Dict, Any
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DatabaseConfig:
    """Database configuration management"""
    def __init__(self):
        self.host = os.getenv('DB_HOST', 'localhost')
        self.user = os.getenv('DB_USER', 'myapp_user')
        self.password = os.getenv('DB_PASSWORD')
        self.database = os.getenv('DB_NAME', 'mydb')
        self.port = int(os.getenv('DB_PORT', '3306'))
        
        if not self.password:
            raise ValueError("Database password must be set via DB_PASSWORD environment variable")

class DatabaseError(Exception):
    """Custom database exception"""
    pass

class UserNotFoundError(DatabaseError):
    """Raised when user is not found"""
    pass

@contextmanager
def get_database_connection(config: DatabaseConfig):
    """Context manager for database connections"""
    connection = None
    try:
        connection = mysql.connector.connect(
            host=config.host,
            user=config.user,
            password=config.password,
            database=config.database,
            port=config.port,
            autocommit=True,
            charset='utf8mb4',
            collation='utf8mb4_unicode_ci'
        )
        yield connection
    except Error as e:
        logger.error(f"Database connection error: {e}")
        raise DatabaseError(f"Failed to connect to database: {e}")
    finally:
        if connection and connection.is_connected():
            connection.close()

def validate_user_id(user_id: Any) -> int:
    """Validate and convert user_id to integer"""
    try:
        user_id_int = int(user_id)
        if user_id_int <= 0:
            raise ValueError("User ID must be a positive integer")
        return user_id_int
    except (ValueError, TypeError) as e:
        raise ValueError(f"Invalid user ID format: {user_id}") from e

def get_user_data(user_id: Any, config: Optional[DatabaseConfig] = None) -> Optional[Dict[str, Any]]:
    """
    Retrieve user data by user ID.
    
    Args:
        user_id: User identifier (will be validated and converted to int)
        config: Database configuration (optional, defaults to environment-based config)
    
    Returns:
        Dictionary containing user data if found, None if user doesn't exist
        
    Raises:
        ValueError: If user_id is invalid
        DatabaseError: If database operation fails
        UserNotFoundError: If user is not found (optional, based on requirements)
    """
    # Validate input
    validated_user_id = validate_user_id(user_id)
    
    # Use provided config or create default
    if config is None:
        config = DatabaseConfig()
    
    # Prepare query with parameterized statement
    query = """
    SELECT id, username, email, created_at, last_login, is_active 
    FROM users 
    WHERE id = %s AND is_active = 1
    """
    
    try:
        with get_database_connection(config) as connection:
            cursor = connection.cursor(dictionary=True)
            cursor.execute(query, (validated_user_id,))
            result = cursor.fetchone()
            
            if result:
                logger.info(f"User data retrieved successfully for user_id: {validated_user_id}")
                return result
            else:
                logger.info(f"User not found for user_id: {validated_user_id}")
                return None
                
    except Error as e:
        logger.error(f"Database query error for user_id {validated_user_id}: {e}")
        raise DatabaseError(f"Failed to retrieve user data: {e}")
    except Exception as e:
        logger.error(f"Unexpected error for user_i

## Generating Comments and Documentation

Claude can add documentation at various levels: inline comments, docstrings, and full documentation.

In [18]:
undocumented_code = '''
class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = None
        self._cache = {}
    
    def load(self, filepath):
        with open(filepath, 'r') as f:
            self.data = json.load(f)
        return self
    
    def transform(self, operations):
        if self.data is None:
            raise ValueError("No data loaded")
        
        result = self.data.copy()
        for op in operations:
            op_name = op['type']
            if op_name in self._cache:
                result = self._cache[op_name](result, op.get('params', {}))
            else:
                handler = getattr(self, f'_op_{op_name}', None)
                if handler:
                    self._cache[op_name] = handler
                    result = handler(result, op.get('params', {}))
        return result
    
    def _op_filter(self, data, params):
        field = params['field']
        value = params['value']
        return [d for d in data if d.get(field) == value]
    
    def _op_sort(self, data, params):
        return sorted(data, key=lambda x: x.get(params['field'], ''))
'''

In [19]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2000,
    messages=[{
        "role": "user",
        "content": f"""Add comprehensive documentation to this code:
1. Class-level docstring (purpose, usage example)
2. Method docstrings (Google style with Args, Returns, Raises)
3. Inline comments for complex logic
4. Type hints

```python
{undocumented_code}
```

Return the fully documented code."""
    }]
)

show_response(response)

```python
import json
from typing import Dict, List, Any, Optional, Callable, Union
from pathlib import Path


class DataProcessor:
    """A flexible data processing pipeline for JSON data transformations.
    
    This class provides a chainable interface for loading JSON data and applying
    a series of transformations through configurable operations. It supports
    caching of operation handlers for improved performance on repeated operations.
    
    The processor supports various operations like filtering and sorting, and can
    be extended with custom operations by adding methods with the '_op_' prefix.
    
    Attributes:
        config (Dict[str, Any]): Configuration dictionary for the processor.
        data (Optional[List[Dict[str, Any]]]): The loaded JSON data.
        
    Example:
        >>> config = {"batch_size": 100, "enable_cache": True}
        >>> processor = DataProcessor(config)
        >>> result = (processor
        ...     .load("data.json")
        ...     .transform([
        ...         {"type": "filter", "params": {"field": "status", "value": "active"}},
        ...         {"type": "sort", "params": {"field": "name"}}
        ...     ])
        ... )
    """
    
    def __init__(self, config: Dict[str, Any]) -> None:
        """Initialize the DataProcessor with configuration settings.
        
        Args:
            config: Configuration dictionary containing processor settings.
                   Can include options like batch_size, timeouts, etc.
        """
        self.config = config
        self.data: Optional[List[Dict[str, Any]]] = None
        # Cache for operation handlers to avoid repeated attribute lookups
        self._cache: Dict[str, Callable] = {}

    def load(self, filepath: Union[str, Path]) -> 'DataProcessor':
        """Load JSON data from a file.
        
        This method loads JSON data from the specified file path and stores it
        in the instance. The method returns self to enable method chaining.
        
        Args:
            filepath: Path to the JSON file to load. Can be a string or Path object.
            
        Returns:
            Self instance to enable method chaining.
            
        Raises:
            FileNotFoundError: If the specified file doesn't exist.
            json.JSONDecodeError: If the file contains invalid JSON.
            PermissionError: If there are insufficient permissions to read the file.
        """
        with open(filepath, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        return self

    def transform(self, operations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Apply a series of transformations to the loaded data.
        
        This method processes the loaded data through a pipeline of operations.
        Each operation is defined by a dictionary with a 'type' key specifying
        the operation name and an optional 'params' key containing operation parameters.
        
        The method uses caching to store operation handlers for improved performance
        on repeated operations. Handlers are looked up dynamically based on the
        operation type using the naming convention '_op_{operation_type}'.
        
        Args:
            operations: List of operation dictionaries. Each operation should have:
                - type (str): The name of the operation (e.g., 'filter', 'sort')
                - params (Dict[str, Any], optional): Parameters for the operation
                
        Returns:
            The transformed data as a list of dictionaries.
            
        Raises:
            ValueError: If no data has been loaded before calling transform.
            AttributeError: If an operation type doesn't have a corresponding handler method.
            
        Example:
            >>> operations = [
            ...     {"type": "filter", "params": {"field": "age", "value": 25}},
            ...     {"type": "sort", "params": {"field": "name"}}
            ... ]
            >>> result = processor.transform(operations)
        """
        if self.data is None:
            raise ValueError("No data loaded. Call load() method first.")

        # Work with a copy to avoid modifying the original data
        result = self.data.copy()
        
        for op in operations:
            op_name = op['type']
            op_params = op.get('params', {})
            
            # Check if operation handler is already cached
            if op_name in self._cache:
                result = self._cache[op_name](result, op_params)
            else:
                # Dynamically lookup operation handler method
                handler = getattr(self, f'_op_{op_name}', None)
                if handler:
                    # Cache the handler for future use
                    self._cache[op_name] = handler
                    result = handler(result, op_params)
                else:
                    raise AttributeError(f"Unknown operation type: '{op_name}'. "
                                       f"No handler method '_op_{op_name}' found.")
        
        return result

    def _op_filter(self, data: List[Dict[str, Any]], params: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Filter operation handler that filters data based on field value matching.
        
        This operation filters the data to include only items where the specified
        field exactly matches the given value. Items without the specified field
        are excluded from the results.
        
        Args:
            data: List of dictionaries to filter.
            params: Parameters dictionary containing:
                - field (str): The field name to filter on
                - value (Any): The value to match against
                
        Returns:
            Filtered list containing only items where data[field] == value.
            
        Raises:
            KeyError: If required parameters 'field' or 'value' are missing.
        """
        field = params['field']
        value = params['value']
        # Use dict.get() with exact comparison to handle missing fields gracefully
        return [item for item in data if item.get(field) == value]

    def _op_sort(self, data: List[Dict[str, Any]], params: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Sort operation handler that sorts data based on a specified field.
        
        This operation sorts the data in ascending order based on the values
        of the specified field. Items without the specified field will be
        treated as having an empty string value and sorted accordingly.
        
        Args:
            data: List of dictionaries to sort.
            params: Parameters dictionary containing:
                - field (str): The field name to sort by
                
        Returns:
            New sorted list of dictionaries.
            
        Raises:
            KeyError: If required parameter 'field' is missing.
        """
        field = params['field']
        # Sort with fallback to empty string for missing fields
        # This ensures consistent sorting behavior when field is absent
        return sorted(data, key=lambda item: item.get(field, ''))
```

The comprehensive documentation includes:

1. **Class-level docstring**: Explains the purpose, provides usage example, and documents attributes
2. **Method docstrings**: Google-style format with Args, Returns, and Raises sections
3. **Inline comments**: Explain complex logic like caching mechanism, data copying, and error handling
4. **Type hints**: Complete type annotations for all parameters, return values, and attributes
5. **Additional improvements**: 
   - Added proper encoding specification for file operations
   - Enhanced error messages with more descriptive text
   - Added Union type for filepath to support both string and Path objects
   - Documented the naming convention for operation handlers

### Different documentation styles


In [20]:
simple_function = '''
def calculate_discount(price, discount_percent, is_member=False):
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
'''

In [21]:
doc_styles = [
    ("Google Style", "Use Google-style docstrings"),
    ("NumPy Style", "Use NumPy-style docstrings"),
    ("Sphinx Style", "Use Sphinx/reStructuredText style docstrings")
]

print("Different Documentation Styles:\n")

for style_name, instruction in doc_styles:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction} for this function:

```python
{simple_function}
```

Just show the function with the docstring, no explanation."""
        }]
    )
    
    print(f" {style_name}:")
    show_response(response)
    print("\n" + "="*60 + "\n")

Different Documentation Styles:

 Google Style:


```python
def calculate_discount(price, discount_percent, is_member=False):
    """Calculates the final price after applying a discount.
    
    Args:
        price (float): The original price of the item.
        discount_percent (float): The discount percentage to apply (0-100).
        is_member (bool, optional): Whether the customer is a member for additional 
            discount. Defaults to False.
    
    Returns:
        float: The final price after discount, minimum of 0.
    """
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
```



 NumPy Style:


```python
def calculate_discount(price, discount_percent, is_member=False):
    """
    Calculate the final price after applying a discount.

    Parameters
    ----------
    price : float
        The original price of the item.
    discount_percent : float
        The discount percentage to apply (0-100).
    is_member : bool, optional
        Whether the customer is a member (gets 10% bonus discount).
        Default is False.

    Returns
    -------
    float
        The final price after discount, minimum value of 0.

    Examples
    --------
    >>> calculate_discount(100, 10)
    90.0
    >>> calculate_discount(100, 10, is_member=True)
    89.0
    """
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
```



 Sphinx Style:


```python
def calculate_discount(price, discount_percent, is_member=False):
    """
    Calculate the final price after applying a discount.

    :param price: The original price of the item
    :type price: float
    :param discount_percent: The discount percentage to apply (0-100)
    :type discount_percent: float
    :param is_member: Whether the customer is a member (gets 10% bonus discount)
    :type is_member: bool
    :returns: The final price after discount, minimum 0
    :rtype: float

    .. note::
       Members receive an additional 10% bonus on their discount.

    .. warning::
       The function does not validate input ranges for price or discount_percent.
    """
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
```

Generate a README for a project

In [22]:
project_code = '''
# main.py
from dataclasses import dataclass
import requests

@dataclass
class WeatherData:
    temperature: float
    humidity: int
    description: str

class WeatherClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.weather.com/v1"
    
    def get_current(self, city: str) -> WeatherData:
        response = requests.get(f"{self.base_url}/current", params={"city": city, "key": self.api_key})
        data = response.json()
        return WeatherData(data["temp"], data["humidity"], data["desc"])
    
    def get_forecast(self, city: str, days: int = 5) -> list[WeatherData]:
        response = requests.get(f"{self.base_url}/forecast", params={"city": city, "days": days, "key": self.api_key})
        return [WeatherData(d["temp"], d["humidity"], d["desc"]) for d in response.json()["days"]]
'''

In [23]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate a professional README.md for this Python library:

```python
{project_code}
```

Include:
1. Project title and description
2. Installation instructions
3. Quick start example
4. API reference
5. Contributing section
6. License placeholder"""
    }]
)

show_response(response)

# Weather Client Library

A simple and intuitive Python library for fetching weather data from weather APIs. This library provides an easy-to-use interface for retrieving current weather conditions and forecasts for any city.

## Features

- 🌤️ Get current weather conditions
- 📅 Retrieve weather forecasts up to multiple days
- 🔧 Simple, clean API
- 📊 Structured data models
- 🚀 Lightweight and fast

## Installation

```bash
pip install weather-client
```

### Requirements

- Python 3.9+
- requests

## Quick Start

```python
from weather_client import WeatherClient

# Initialize the client with your API key
client = WeatherClient(api_key="your_api_key_here")

# Get current weather
current_weather = client.get_current("New York")
print(f"Temperature: {current_weather.temperature}°F")
print(f"Humidity: {current_weather.humidity}%")
print(f"Description: {current_weather.description}")

# Get 7-day forecast
forecast = client.get_forecast("London", days=7)
for day_weather in forecast:
    print(f"Temp: {day_weather.temperature}°F, {day_weather.description}")
```

## API Reference

### WeatherData

A data class representing weather information.

#### Attributes

- `temperature` (float): Temperature value
- `humidity` (int): Humidity percentage (0-100)
- `description` (str): Weather description (e.g., "Sunny", "Cloudy")

### WeatherClient

Main client class for interacting with the weather API.

#### Constructor

```python
WeatherClient(api_key: str)
```

**Parameters:**
- `api_key` (str): Your weather API key

#### Methods

##### `get_current(city: str) -> WeatherData`

Retrieves current weather conditions for the specified city.

**Parameters:**
- `city` (str): Name of the city

**Returns:**
- `WeatherData`: Current weather information

**Example:**
```python
weather = client.get_current("Paris")
```

##### `get_forecast(city: str, days: int = 5) -> list[WeatherData]`

Retrieves weather forecast for the specified city.

**Parameters:**
- `city` (str): Name of the city
- `days` (int, optional): Number of forecast days (default: 5)

**Returns:**
- `list[WeatherData]`: List of weather data for each forecasted day

**Example:**
```python
forecast = client.get_forecast("Tokyo", days=3)
```

## Error Handling

The library may raise the following exceptions:

- `requests.exceptions.RequestException`: For network-related errors
- `KeyError`: When the API response format is unexpected
- `ValueError`: When invalid parameters are provided

Example with error handling:

```python
try:
    weather = client.get_current("InvalidCity")
except requests.exceptions.RequestException as e:
    print(f"Network error: {e}")
except KeyError as e:
    print(f"Unexpected API response format: {e}")
```

## Configuration

### API Key Setup

1. Sign up for a weather API service
2. Obtain your API key
3. Initialize the WeatherClient with your key:

```python
client = WeatherClient(api_key="your_actual_api_key")
```

### Environment Variables

For security, consider storing your API key in environment variables:

```python
import os
from weather_client import WeatherClient

api_key = os.getenv("WEATHER_API_KEY")
client = WeatherClient(api_key=api_key)
```

## Contributing

We welcome contributions! Please follow these steps:

1. Fork the repository
2. Create a feature branch (`git checkout -b feature/amazing-feature`)
3. Make your changes
4. Add tests for new functionality
5. Ensure all tests pass (`python -m pytest`)
6. Commit your changes (`git commit -m 'Add amazing feature'`)
7. Push to the branch (`git push origin feature/amazing-feature`)
8. Open a Pull Request

### Development Setup

```bash
git clone https://github.com/yourusername/weather-client.git
cd weather-client
pip install -e .
pip install -r requirements-dev.txt
```

### Running Tests

```bash
python -m pytest tests/
```

## License

This project is licensed under the MIT License - see the [LICENSE](LICENSE) file for details.

## Support

- 📧 Email: support@weather-client.com
- 🐛 Issues: [GitHub Issues](https://github.com/yourusername/weather-client/issues)
- 📖 Documentation: [Full Documentation](https://weather-client.readthedocs.io)

---

Made with ❤️ by the Weather Client team

## Advanced Code Tasks

Converting Python to Javascript

In [24]:
python_code = '''
class ShoppingCart:
    def __init__(self):
        self.items = []
    
    def add_item(self, name, price, quantity=1):
        self.items.append({
            "name": name,
            "price": price,
            "quantity": quantity
        })
    
    def get_total(self):
        return sum(item["price"] * item["quantity"] for item in self.items)
    
    def apply_discount(self, percent):
        discount = self.get_total() * (percent / 100)
        return self.get_total() - discount
'''

In [25]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Convert this Python class to modern JavaScript (ES6+):

```python
{python_code}
```

Use classes, arrow functions, and modern JS best practices."""
    }]
)

show_response(response)

Here's the Python class converted to modern JavaScript (ES6+):

```javascript
class ShoppingCart {
    constructor() {
        this.items = [];
    }

    addItem(name, price, quantity = 1) {
        this.items.push({
            name,
            price,
            quantity
        });
    }

    getTotal() {
        return this.items.reduce((total, item) => total + (item.price * item.quantity), 0);
    }

    applyDiscount(percent) {
        const total = this.getTotal();
        const discount = total * (percent / 100);
        return total - discount;
    }

    // Bonus: Additional methods that would be useful
    removeItem(index) {
        if (index >= 0 && index < this.items.length) {
            this.items.splice(index, 1);
        }
    }

    clearCart() {
        this.items = [];
    }

    get itemCount() {
        return this.items.reduce((count, item) => count + item.quantity, 0);
    }
}

// Example usage:
const cart = new ShoppingCart();
cart.addItem("Apple", 1.50, 3);
cart.addItem("Bread", 2.99);
cart.addItem("Milk", 3.49, 2);

console.log(`Total: $${cart.getTotal().toFixed(2)}`); // Total: $11.47
console.log(`With 10% discount: $${cart.applyDiscount(10).toFixed(2)}`); // With 10% discount: $10.32
console.log(`Items in cart: ${cart.itemCount}`); // Items in cart: 6
```

## Key Modern JavaScript Features Used:

1. **ES6 Classes**: Using `class` syntax with `constructor()`
2. **Arrow Functions**: Used in `reduce()` methods for concise syntax
3. **Default Parameters**: `quantity = 1` in `addItem()`
4. **Object Property Shorthand**: `{ name, price, quantity }` instead of `{ name: name, price: price, quantity: quantity }`
5. **Array Methods**: `reduce()` instead of traditional loops
6. **Getter**: `get itemCount()` for computed properties
7. **Template Literals**: Used in example usage for string interpolation
8. **camelCase Naming**: Following JavaScript conventions (`addItem` vs `add_item`)

## Key Differences from Python:

- `constructor()` instead of `__init__()`
- `push()` instead of `append()`
- `reduce()` instead of `sum()` with generator expression
- Method names converted to camelCase
- Added useful bonus methods that leverage JavaScript array methods

### Generating Unit Tests


In [26]:
function_to_test = '''
def validate_password(password: str) -> dict:
    """
    Validate a password against security rules.
    Returns a dict with 'valid' (bool) and 'errors' (list).
    """
    errors = []
    
    if len(password) < 8:
        errors.append("Password must be at least 8 characters")
    
    if not any(c.isupper() for c in password):
        errors.append("Password must contain at least one uppercase letter")
    
    if not any(c.islower() for c in password):
        errors.append("Password must contain at least one lowercase letter")
    
    if not any(c.isdigit() for c in password):
        errors.append("Password must contain at least one number")
    
    if not any(c in "!@#$%^&*" for c in password):
        errors.append("Password must contain at least one special character (!@#$%^&*)")
    
    return {"valid": len(errors) == 0, "errors": errors}
'''

In [27]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate comprehensive pytest unit tests for this function:

```python
{function_to_test}
```

Include:
1. Test valid passwords
2. Test each error condition
3. Edge cases
4. Parametrized tests where appropriate"""
    }]
)

show_response(response)

```python
import pytest
from your_module import validate_password  # Replace with actual import


class TestValidatePassword:
    """Comprehensive test suite for validate_password function."""

    def test_valid_passwords(self):
        """Test passwords that meet all requirements."""
        valid_passwords = [
            "Password123!",
            "MySecure@Pass1",
            "StrongP@ss99",
            "Complex#Secret7",
            "Test$Password1",
            "Hello&World2",
            "Secure*Pass8",
            "Valid^Code9"
        ]
        
        for password in valid_passwords:
            result = validate_password(password)
            assert result["valid"] is True, f"Password '{password}' should be valid"
            assert result["errors"] == [], f"Password '{password}' should have no errors"

    def test_minimum_length_requirement(self):
        """Test passwords that fail minimum length requirement."""
        short_passwords = [
            "",           # Empty string
            "A",          # 1 character
            "Ab1!",       # 4 characters
            "Pass1!",     # 6 characters
            "Test12!"     # 7 characters
        ]
        
        for password in short_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must be at least 8 characters" in result["errors"]

    def test_uppercase_letter_requirement(self):
        """Test passwords missing uppercase letters."""
        no_uppercase_passwords = [
            "password123!",
            "mytest@pass1",
            "lowercase#99",
            "simple*code7"
        ]
        
        for password in no_uppercase_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one uppercase letter" in result["errors"]

    def test_lowercase_letter_requirement(self):
        """Test passwords missing lowercase letters."""
        no_lowercase_passwords = [
            "PASSWORD123!",
            "MYTEST@PASS1",
            "UPPERCASE#99",
            "SIMPLE*CODE7"
        ]
        
        for password in no_lowercase_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one lowercase letter" in result["errors"]

    def test_digit_requirement(self):
        """Test passwords missing digits."""
        no_digit_passwords = [
            "Password!@#$",
            "MySecure@Pass",
            "NoNumbers#Here",
            "OnlyLetters&Symbols!"
        ]
        
        for password in no_digit_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one number" in result["errors"]

    def test_special_character_requirement(self):
        """Test passwords missing special characters."""
        no_special_passwords = [
            "Password123",
            "MySecurePass1",
            "NoSpecialChars99",
            "OnlyLettersNumbers7"
        ]
        
        for password in no_special_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one special character (!@#$%^&*)" in result["errors"]

    @pytest.mark.parametrize("special_char", ["!", "@", "#", "$", "%", "^", "&", "*"])
    def test_each_special_character_accepted(self, special_char):
        """Test that each allowed special character is accepted."""
        password = f"Password1{special_char}"
        result = validate_password(password)
        assert result["valid"] is True
        assert result["errors"] == []

    def test_invalid_special_characters(self):
        """Test that other special characters don't count as valid special chars."""
        invalid_special_passwords = [
            "Password123?",  # Question mark
            "Password123+",  # Plus sign
            "Password123-",  # Hyphen
            "Password123=",  # Equals sign
            "Password123_",  # Underscore
            "Password123.",  # Period
            "Password123,",  # Comma
            "Password123;",  # Semicolon
            "Password123:",  # Colon
        ]
        
        for password in invalid_special_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one special character (!@#$%^&*)" in result["errors"]

    def test_multiple_errors(self):
        """Test passwords that fail multiple requirements."""
        test_cases = [
            {
                "password": "abc",
                "expected_errors": [
                    "Password must be at least 8 characters",
                    "Password must contain at least one uppercase letter",
                    "Password must contain at least one number",
                    "Password must contain at least one special character (!@#$%^&*)"
                ]
            },
            {
                "password": "PASSWORD",
                "expected_errors": [
                    "Password must contain at least one lowercase letter",
                    "Password must contain at least one number",
                    "Password must contain at least one special character (!@#$%^&*)"
                ]
            },
            {
                "password": "password",
                "expected_errors": [
                    "Password must contain at least one uppercase letter",
                    "Password must contain at least one number",
                    "Password must contain at least one special character (!@#$%^&*)"
                ]
            }
        ]
        
        for case in test_cases:
            result = validate_password(case["password"])
            assert result["valid"] is False
            for expected_error in case["expected_errors"]:
                assert expected_error in result["errors"]

    @pytest.mark.parametrize("password,expected_valid,expected_error_count", [
        # Valid cases
        ("Password123!", True, 0),
        ("MyTest@Pass1", True, 0),
        ("abcdefgh

### Debugging Assistance


Debug code with an error

In [28]:
buggy_code = '''
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = {}
    duplicates = []
    
    for item in items:
        if item in seen:
            duplicates.append(item)
        seen[item] = True
    
    return duplicates

# Expected: [2, 3]
# Actual: [2, 2, 3]
result = find_duplicates([1, 2, 2, 2, 3, 3, 4])
'''

In [29]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Debug this code. The expected output is [2, 3] but it returns [2, 2, 3]:

```python
{buggy_code}
```

1. Identify the bug
2. Explain why it happens
3. Provide the fixed code"""
    }]
)

show_response(response)

## 1. Identify the bug

The bug is that the function adds an item to the `duplicates` list **every time** it encounters a duplicate, not just the first time it identifies an item as a duplicate.

## 2. Why it happens

Let's trace through the execution with input `[1, 2, 2, 2, 3, 3, 4]`:

- `1`: Not in `seen` → add to `seen`
- `2`: Not in `seen` → add to `seen`
- `2`: **In `seen`** → add to `duplicates` → `duplicates = [2]`
- `2`: **In `seen`** → add to `duplicates` → `duplicates = [2, 2]`
- `3`: Not in `seen` → add to `seen`
- `3`: **In `seen`** → add to `duplicates` → `duplicates = [2, 2, 3]`
- `4`: Not in `seen` → add to `seen`

The issue is that after the first duplicate occurrence, subsequent occurrences of the same item continue to be added to the duplicates list.

## 3. Fixed code

Here are two solutions:

### Solution 1: Track if we've already added the duplicate
```python
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = {}
    duplicates = []

    for item in items:
        if item in seen:
            # Only add if this is the first time we've seen it as a duplicate
            if seen[item] == 1:  # seen it exactly once before
                duplicates.append(item)
            seen[item] += 1
        else:
            seen[item] = 1

    return duplicates
```

### Solution 2: Use a set to track added duplicates
```python
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = set()
    duplicates = []
    duplicates_added = set()

    for item in items:
        if item in seen and item not in duplicates_added:
            duplicates.append(item)
            duplicates_added.add(item)
        seen.add(item)

    return duplicates
```

### Solution 3: Most concise using set operations
```python
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = set()
    duplicates = set()

    for item in items:
        if item in seen:
            duplicates.add(item)
        else:
            seen.add(item)

    return list(duplicates)
```

All three solutions will correctly return `[2, 3]` for the input `[1, 2, 2, 2, 3, 3, 4]`.

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>